In [1]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(
            to right,
            limegreen,
            deepskyblue,
            limegreen
        );
    }
</style>

# Creating Agents with the OpenAI Agents SDK
---
# MCP (Model Context Protocol): Hosted Weather and Geocoding Servers
* **MCP (Model Context Protocol)**
    * Open standard created by Anthropic for connecting AI models to external tools and data sources
    * MCP servers expose tool schemas and tool calls to agent clients
* **Hosted MCP servers**
    * Hosted in the cloud
    * Connect via a URL
    * Single server can host multiple capabilities
* **`MCPServerStreamableHttp`**
    * Connects to a remote MCP endpoint over HTTPS
* **Local MCP server**
    * Runs on your machine, often as a subprocess
    * Useful when the tools need local files, databases, or private resources
* **`mcp_servers=`**
    * Parameter on `Agent`
    * Tools from connected MCP servers and `tools=` entries are all available to the model as tools
* **`cache_tools_list=True`**
    * Fetches the tool list once and reuses it across turns
---


## Imports

In [2]:
import os
from agents import Agent, trace
from agents.mcp import MCPServerStreamableHttp
from source.agent_loop import run_conversation

## MCP Server URL
* Server: **`https://weather.chukai.io/mcp`**
    * Open-Meteo weather APIs
    * Offers worldwide coverage, built-in geocoding, no API key
* Free public servers like this one may **sleep when idle**
    * The first connection of the day can take longer than the SDK's 5-second default timeout
    * We allow up to 60 seconds, and retry a failed tool call up to twice

In [3]:
WEATHER_MCP_URL = 'https://weather.chukai.io/mcp'

# Free public MCP servers may sleep when idle, so the first connection of the
# day can take longer than the SDK's 5-second default timeouts. Allow more time.
MCP_TIMEOUT_SECONDS = 60

## Inspect the Available Tools
* `MCPServerStreamableHttp` fetches tool schemas over HTTPS — no subprocess involved

In [4]:
async with MCPServerStreamableHttp(
    params={
        'url': WEATHER_MCP_URL,
        'timeout': MCP_TIMEOUT_SECONDS # HTTP request timeout
    },
    client_session_timeout_seconds=MCP_TIMEOUT_SECONDS, # MCP session timeout
    max_retry_attempts=2, # retry a failed list_tools or tool call
    cache_tools_list=True
) as server:
    tools = await server.list_tools()

print(f'{len(tools)} tools:\n')
for t in tools:
    props = t.input_schema.get('properties', {})
    req   = set(t.input_schema.get('required', []))
    print(f'  {t.name}')
    print(f'    {t.description.splitlines()[0]}')
    for param, schema in props.items():
        marker = ' (required)' if param in req else ' (optional)'
        desc   = schema.get('description', '')[:80]
        print(f'    param: {param}{marker} — {desc}')
    print()

12 tools:

  get_air_quality
    Get air quality data and forecasts from Open-Meteo Air Quality API.
    param: latitude (required) — 
    param: longitude (required) — 
    param: timezone (optional) — 
    param: hourly (optional) — 
    param: domains (optional) — 

  batch_get_air_quality
    Get air quality data for multiple locations in a single API call.
    param: latitudes (required) — 
    param: longitudes (required) — 
    param: timezone (optional) — 
    param: hourly (optional) — 
    param: domains (optional) — 

  get_weather_forecast
    Get comprehensive weather forecast with current conditions, hourly, and daily forecasts.
    param: latitude (required) — 
    param: longitude (required) — 
    param: temperature_unit (optional) — 
    param: wind_speed_unit (optional) — 
    param: precipitation_unit (optional) — 
    param: timezone (optional) — 
    param: forecast_days (optional) — 
    param: current_weather (optional) — 
    param: hourly (optional) — 
    par

## Build and Run the Weather Agent


In [5]:
INSTRUCTIONS = """
You are a weather assistant with worldwide coverage.

The server provides both geocoding and weather tools. When the user gives a city or place
name, use the geocoding tool first to get coordinates, then pass them to the weather tools.

Always format your responses as Markdown:
- Present forecasts and observations as Markdown tables
- Use bold for key values (temperature, wind, alerts)
- Report temperatures in both °F and °C
- Mention active alerts when relevant
"""

async with MCPServerStreamableHttp(
    params={
        'url': WEATHER_MCP_URL,
        'timeout': MCP_TIMEOUT_SECONDS # HTTP request timeout
    },
    client_session_timeout_seconds=MCP_TIMEOUT_SECONDS, # MCP session timeout
    max_retry_attempts=2, # retry a failed list_tools or tool call
    cache_tools_list=True
) as server:
    agent = Agent(
        name='WeatherAssistant',
        model='gpt-6.1-terra',
        instructions=INSTRUCTIONS,
        mcp_servers=[server]
    )
    with trace('02-05-05-hosted-mcp-weather-primary'):
        await run_conversation(
            'Weather assistant ready. Ask about forecasts, conditions, or alerts anywhere in the world!',
            agent
        )

Weather assistant ready. Ask about forecasts, conditions, or alerts anywhere in the world!



Input [1]:  Give me the 7-day forecast for Miami


### Miami, Florida: 7-day forecast

| Date | Conditions | High | Low | Rain chance | Max wind |
|---|---|---:|---:|---:|---:|
| Mon, Nov 2 | Light drizzle | **82°F (28°C)** | **75°F (24°C)** | 37% | **15 mph** |
| Tue, Nov 3 | Overcast | **81°F (27°C)** | **76°F (24°C)** | 38% | **13 mph** |
| Wed, Nov 4 | Light drizzle | **83°F (28°C)** | **75°F (24°C)** | 44% | **13 mph** |
| Thu, Nov 5 | Light drizzle | **83°F (28°C)** | **79°F (26°C)** | 18% | **12 mph** |
| Fri, Nov 6 | Overcast | **81°F (27°C)** | **78°F (25°C)** | 3% | **11 mph** |
| Sat, Nov 7 | Light drizzle | **84°F (29°C)** | **76°F (25°C)** | 5% | **10 mph** |
| Sun, Nov 8 | Partly cloudy | **84°F (29°C)** | **76°F (25°C)** | 13% | **14 mph** |

*Source: Open-Meteo. Times and dates are local. The forecast data does not include active weather alerts; check the National Weather Service for those.*


Input [2]:  exit


User terminated app.


## Sample Prompts

* What's the current weather in Tokyo?
* Give me the 7-day forecast for Miami
* Are there any weather alerts in Oklahoma City?
* Compare current conditions in Seattle, Phoenix, and Chicago
* What are local forecasters saying about Austin this week?
* What do the NWS meteorologists say about Boston this week?
* What's the hourly forecast for New York City today?

## Fallback — If `weather.chukai.io` Is Down
* Run this cell instead of the one above
* Uses two separate hosted servers:
    * `geocoder.chukai.io` for geocoding 
    * `nws.caseyjhand.com` for US weather data

In [ ]:
from agents.mcp import MCPServerStreamableHttp

NWS_MCP_URL = 'https://nws.caseyjhand.com/mcp' # US only
GEOCODER_MCP_URL = 'https://geocoder.chukai.io/mcp' # IBM/chuk-mcp-geocoder

INSTRUCTIONS = """
You are a United States weather assistant.

Tools available:
- geocode (from geocoder server): city or place name → lat/lon
- batch_geocode: geocode multiple cities in one call
- NWS weather tools: forecasts, observations, alerts (require lat/lon)

Always format your responses as Markdown:
- Present forecasts and observations as Markdown tables
- Use bold for key values (temperature, wind, alerts)
- Report temperatures in both °F and °C
- Mention active alerts when relevant
"""

# parenthesized context manager in the `with` statement enables 
# managing a comma-separated list of MCP servers in this example
async with ( 
    MCPServerStreamableHttp(
        params={'url': GEOCODER_MCP_URL, 'timeout': MCP_TIMEOUT_SECONDS},
        client_session_timeout_seconds=MCP_TIMEOUT_SECONDS,
        max_retry_attempts=2,
        cache_tools_list=True,
    ) as geo_server,
    MCPServerStreamableHttp(
        params={'url': NWS_MCP_URL, 'timeout': MCP_TIMEOUT_SECONDS},
        client_session_timeout_seconds=MCP_TIMEOUT_SECONDS,
        max_retry_attempts=2,
        cache_tools_list=True,
    ) as nws_server
):
    agent = Agent(
        name='WeatherAssistant',
        model='gpt-6.1-terra',
        instructions=INSTRUCTIONS,
        mcp_servers=[geo_server, nws_server] # two MCP servers
    )
    with trace('02-05-05-hosted-mcp-weather-fallback'):
        await run_conversation(
            'Weather assistant ready (fallback mode — US locations only).',
            agent,
        )

Weather assistant ready (fallback mode — US locations only).


---

## Hosted MCP with Authentication
* Production MCP servers typically require authentication
* Pass credentials via the `headers` key inside `params` dictionary
* https://developers.openai.com/plugins/build/auth

---

## MCP Servers vs. `@function_tool`
* Both expose tools to an agent
* Choice between them is **where the tool code runs and what can reach it**
* **Trade-off**: in-process vs. out-of-process

    | | `@function_tool` | Local MCP (`MCPServerStdio`) | Hosted MCP (`MCPServerStreamableHttp`) |
    |---|---|---|---|
    | Tool code runs | In your Python process | Separate subprocess | Remote server in the cloud |
    | What can use it | This agent only | Any MCP-compatible client on this machine | Any MCP-compatible client anywhere |
    | Setup | Decorate a function | Code + launch a server | Just provide a URL |
    | Latency | Lowest (direct call) | Low (local pipe) | Higher (HTTPS round-trip) |

### When to choose each 
* **`@function_tool`** 
    * Tools are specific to a given agent/app
    * Don't need to share them with other agents or non-Python clients
* **Local MCP server**
    * Reuse the same tools across multiple agents
    * Share with other MCP-compatible clients (Claude Desktop, VS Code, etc.) **on the same machine**
* **Hosted MCP Server**
    * Want the tools accessible from anywhere without distributing code
    * Examples
        * Shared team database server
        * Public API wrapper
        * Any capability that benefits from being centrally deployed

---

© 2026 by Deitel & Associates, Inc. All Rights Reserved.